<div style='text-align:center;padding:30px;background:linear-gradient(135deg,#0f0c29,#302b63,#24243e);border-radius:15px;margin:10px 0'>
  <h1 style='color:#fff;margin:0 0 8px 0;font-size:2.2em'>🎥 LTX-Video Generator</h1>
  <h3 style='color:#c0c0ff;margin:0 0 5px 0;font-weight:400'>Lightning AI Studio | Wan2GP + mmgp</h3>
  <p style='color:#aaa;margin:0'>Text-to-Video & Image-to-Video | Q4_K_M GGUF</p>
</div>

<p align="center">
  <a href="https://www.youtube.com/@thebuildai?sub_confirmation=1"><img src="https://img.shields.io/badge/YouTube-SUBSCRIBE-red?style=for-the-badge&logo=youtube&logoColor=white"> /img</a>
  <a href="https://www.instagram.com/thebuildai/"><img src="https://img.shields.io/badge/Instagram-FOLLOW-E4405F?style=for-the-badge&logo=instagram&logoColor=white"> /img</a>
  <a href="https://www.tiktok.com/@the.build.ai"><img src="https://img.shields.io/badge/TikTok-FOLLOW-000000?style=for-the-badge&logo=tiktok&logoColor=white"> /img</a>
  <a href="https://github.com/cafermutluozkan"><img src="https://img.shields.io/badge/GitHub-FOLLOW-181717?style=for-the-badge&logo=github&logoColor=white"> /img</a>
</p>

---


### 🚀 Quick Start
1. **GPU Studio → L4** (22 GB VRAM)
2. **Cell 1** = setup + download (~10 min)
3. **Cell 2** = verify models (always run after Cell 1)
4. **Cell 3** = launch Gradio

> ⚠️ Re-upload this notebook if you see old line numbers in error traces

In [ ]:
# @title ⚙️ Cell 1 — Setup & Download (~17.8 GB)
import os, sys, subprocess
from pathlib import Path

HOME = os.path.expanduser('~')

# ── 1/4  Base deps ────────────────────────────────────────────────────────────
print('[1/4] Installing base dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'huggingface_hub', 'gradio', 'gguf', 'soundfile'], check=True)

import torch
print(f'  PyTorch {torch.__version__} | CUDA {torch.version.cuda}')

# ── 2/4  Clone & install Wan2GP ───────────────────────────────────────────────
print('[2/4] Cloning Wan2GP...')
os.chdir(HOME)
if not os.path.exists('Wan2GP'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/deepbeepmeep/Wan2GP.git'], check=True)
os.chdir('Wan2GP')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mmgp'], check=True)

# ── 3/4  Fix torchaudio ───────────────────────────────────────────────────────
# Lightning.ai stacks two conda envs; system torchaudio can be ABI-incompatible
# or partially overwritten, causing 'undefined symbol' / 'dropping_io_support'.
# Full uninstall + cache purge + fresh install from the official PyTorch index.
_tv       = torch.__version__.split('+')[0]           # e.g. '2.5.1'
_cuda_tag = 'cu' + torch.version.cuda.replace('.', '')[:3]  # e.g. 'cu121'
print(f'[3/4] Fixing torchaudio=={_tv} for {_cuda_tag}...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'torchaudio', '-y'],
               capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'],
               capture_output=True)
_r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q',
     f'torchaudio=={_tv}',
     '--extra-index-url', f'https://download.pytorch.org/whl/{_cuda_tag}'],
    capture_output=True, text=True)
if _r.returncode != 0:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q',
         'torchaudio',
         '--extra-index-url', f'https://download.pytorch.org/whl/{_cuda_tag}'],
        check=False)
    print('  torchaudio installed (fallback, unpinned)')
else:
    print(f'  torchaudio=={_tv} OK')

# ── 4/4  Download models ──────────────────────────────────────────────────────
print('[4/4] Downloading models...')
from huggingface_hub import hf_hub_download

MODEL_DIR = f'{HOME}/Wan2GP/models'

def dl(repo, filename, dest_dir, dest_name=None):
    """Download a single file from HuggingFace Hub into dest_dir.
    dest_name lets you rename the file on disk (avoids double-nesting
    when filename contains a subfolder prefix).
    """
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    fp = os.path.join(dest_dir, dest_name or os.path.basename(filename))
    if not os.path.exists(fp):
        print(f'  Downloading {os.path.basename(fp)}...')
        hf_hub_download(repo_id=repo, filename=filename,
                        local_dir=dest_dir, local_dir_use_symlinks=False)
        # hf_hub_download preserves the in-repo subfolder inside local_dir;
        # rename to the flat destination path.
        tmp = os.path.join(dest_dir, filename)
        if os.path.exists(tmp) and tmp != fp:
            os.makedirs(os.path.dirname(fp), exist_ok=True)
            os.rename(tmp, fp)
    else:
        print(f'  ✓ {os.path.basename(fp)}')

# GGUF Transformer
dl('Abiray/LTX-2.3-22B-DISTILLED-1.1-GGUF',
   'LTX-2.3-22B-distilled-1.1-Q4_K_M.gguf',
   MODEL_DIR,
   'ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf')

# Companion safetensors (flat files in repo root)
for _f in [
    'ltx-2.3-22b-distilled-lora-384.safetensors',
    'ltx-2.3-22b_embeddings_connector.safetensors',
    'ltx-2.3-22b_text_embedding_projection.safetensors',
    'ltx-2.3-22b_vae.safetensors',
    'ltx-2.3-22b_audio_vae.safetensors',
    'ltx-2.3-22b_vocoder.safetensors',
    'ltx-2.3-spatial-upscaler-x2-1.1.safetensors',
    'ltx-2.3-temporal-upscaler-x2-1.0.safetensors',
]:
    dl('DeepBeepMeep/LTX-2', _f, MODEL_DIR)

# Gemma text encoder (files live inside a subfolder in the repo)
GEMMA     = 'gemma-3-12b-it-qat-q4_0-unquantized'
GEMMA_DIR = f'{MODEL_DIR}/{GEMMA}'
for _gf in [
    'gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors',
    'added_tokens.json', 'chat_template.json', 'config_light.json',
    'generation_config.json', 'preprocessor_config.json',
    'processor_config.json', 'special_tokens_map.json',
    'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json',
]:
    # Pass dest_name=_gf so the file lands directly in GEMMA_DIR
    # (not double-nested as GEMMA_DIR/GEMMA/file)
    dl('DeepBeepMeep/LTX-2', f'{GEMMA}/{_gf}', GEMMA_DIR, dest_name=_gf)

print('\n✓ All done — run Cell 2 next.')


In [ ]:
# @title 🔍 Cell 2 — Verify & Fix Models (always run this before Cell 3)
import os, glob
from pathlib import Path
from huggingface_hub import hf_hub_download

HOME      = os.path.expanduser('~')
MODEL_DIR = f'{HOME}/Wan2GP/models'
GEMMA     = 'gemma-3-12b-it-qat-q4_0-unquantized'
GEMMA_DIR = f'{MODEL_DIR}/{GEMMA}'
GEMMA_FILE = 'gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors'

def _find_gemma(model_dir, gemma_dir, gemma_file):
    """Search for the Gemma safetensors in multiple likely locations."""
    candidates = [
        # Correct flat location (dest_name= fix applied)
        os.path.join(gemma_dir, gemma_file),
        # Double-nested (old dl() without dest_name)
        os.path.join(gemma_dir, GEMMA, gemma_file),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    # Wide recursive search as last resort
    hits = glob.glob(f'{model_dir}/**/{gemma_file}', recursive=True)
    if hits:
        return sorted(hits)[0]
    # Any gemma quanto file
    hits = glob.glob(f'{model_dir}/**/*gemma*quanto*.safetensors', recursive=True)
    if hits:
        return sorted(hits)[0]
    return None

# ── Search ────────────────────────────────────────────────────────────────────
text_encoder_file = _find_gemma(MODEL_DIR, GEMMA_DIR, GEMMA_FILE)

if text_encoder_file is None:
    print('Gemma safetensors not found — downloading now...')
    Path(GEMMA_DIR).mkdir(parents=True, exist_ok=True)
    hf_hub_download(
        repo_id='DeepBeepMeep/LTX-2',
        filename=f'{GEMMA}/{GEMMA_FILE}',
        local_dir=GEMMA_DIR,
        local_dir_use_symlinks=False,
    )
    # hf_hub_download nests the file; move it to the flat location
    nested = os.path.join(GEMMA_DIR, GEMMA, GEMMA_FILE)
    flat   = os.path.join(GEMMA_DIR, GEMMA_FILE)
    if os.path.exists(nested) and not os.path.exists(flat):
        os.rename(nested, flat)
    text_encoder_file = flat

print(f'✓ text_encoder_file = {text_encoder_file}')

# ── Transformer ───────────────────────────────────────────────────────────────
transformer_path = os.path.join(MODEL_DIR, 'ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf')
if not os.path.exists(transformer_path):
    raise FileNotFoundError(f'Transformer GGUF not found: {transformer_path}\nRe-run Cell 1.')
print(f'✓ transformer      = {os.path.basename(transformer_path)}')

# ── Quick sanity: list everything in MODEL_DIR ────────────────────────────────
print('\nModel directory:')
for entry in sorted(os.scandir(MODEL_DIR), key=lambda e: e.name):
    size = os.path.getsize(entry.path) / 1024**3 if entry.is_file() else 0
    tag  = f'{size:.2f} GB' if entry.is_file() else '<dir>'
    print(f'  {entry.name:60s} {tag}')

print('\n✓ Ready — run Cell 3 to launch Gradio.')


In [ ]:
# @title 🚀 Cell 3 — Launch Gradio
# Requires Cell 2 to have been run first (sets text_encoder_file, transformer_path)
import os, sys, gc, json, random, glob, traceback, time
from pathlib import Path
from datetime import datetime

HOME      = os.path.expanduser('~')
WAN2GP    = f'{HOME}/Wan2GP'
MODEL_DIR = f'{WAN2GP}/models'

os.chdir(WAN2GP)
sys.path.insert(0, WAN2GP)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'

import torch
import gradio as gr
from shared.utils.audio_video import save_video

print(f'GPU:  {torch.cuda.get_device_name()}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

# ── GGUF Handler ──
import shared.qtypes.gguf

# ── Patch config loading ──────────────────────────────────────────────────────
import models.ltx2.ltx2 as ltx2_mod

def _patched_load(path, fallback_config_path=None):
    from mmgp import quant_router
    if isinstance(path, (list, tuple)):
        path = path[0] if path else ''
    if not path:
        return {}
    try:
        _, metadata = quant_router.load_metadata_state_dict(path)
        if metadata:
            config_raw = metadata.get('config')
            if config_raw:
                config = ltx2_mod._normalize_config(config_raw)
                if config:
                    return config
    except Exception:
        pass
    return {}

ltx2_mod._load_config_from_checkpoint = _patched_load

# ── Resolve model paths (Cell 2 sets these; fall back to defaults) ────────────
try:
    _te = text_encoder_file   # set by Cell 2
    _tr = transformer_path    # set by Cell 2
except NameError:
    # Cell 2 was skipped — try to find the files anyway
    _GEMMA     = 'gemma-3-12b-it-qat-q4_0-unquantized'
    _GEMMA_DIR = f'{MODEL_DIR}/{_GEMMA}'
    _GEMMA_FILE = 'gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors'
    _hits = (glob.glob(f'{_GEMMA_DIR}/*.safetensors') +
             glob.glob(f'{_GEMMA_DIR}/**/*.safetensors', recursive=True) +
             glob.glob(f'{MODEL_DIR}/**/*gemma*quanto*.safetensors', recursive=True))
    if not _hits:
        raise FileNotFoundError('Gemma safetensors not found. Run Cell 2 first.')
    _quanto = [f for f in _hits if 'quanto' in f]
    _te = (_quanto or _hits)[0]
    _tr = f'{MODEL_DIR}/ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf'
    print(f'(Cell 2 skipped) text_encoder_file = {_te}')

# ── Load Model ────────────────────────────────────────────────────────────────
print(f'\nLoading LTX-2.3 22B Distilled 1.1 (Q4_K_M)...')
print(f'  transformer  : {os.path.basename(_tr)}')
print(f'  text encoder : {os.path.basename(_te)}')

from mmgp import offload
from shared.utils import files_locator as fl
fl.set_checkpoints_paths(['models', 'ckpts', '.'])
from models.ltx2.ltx2_handler import family_handler

base_model_type = 'ltx2_22B'
model_def = {'ltx2_pipeline': 'distilled'}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

ltx2_model, pipe = family_handler.load_model(
    model_filename=_tr,
    model_type='ltx2_22B_distilled',
    base_model_type=base_model_type,
    model_def=model_def,
    dtype=torch.float16,
    VAE_dtype=torch.float16,
    text_encoder_filename=_te,
)

# ── mmgp Profile 4 ────────────────────────────────────────────────────────────
print('Applying mmgp Profile 4...')
offload.profile(
    pipe, profile_no=4, quantizeTransformer=False,
    convertWeightsFloatTo=torch.float16,
    budgets={
        'transformer': 6000, 'text_encoder': 1500,
        'video_encoder': 2000, 'video_decoder': 3000,
        'audio_encoder': 1000, 'audio_decoder': 1000,
        'vocoder': 500,  'spatial_upsampler': 1500,
        'vae': 1000,     '*': 1000,
    },
)
offload.shared_state['_attention'] = 'sdpa'
print('✓ Model ready')

OUTPUT_DIR = Path(f'{HOME}/videos')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────────────
def get_res(base_res, aspect):
    bases  = {'1080p': 1088, '720p': 704, '540p': 544, '480p': 480}
    ratios = {'16:9': 16/9, '4:3': 4/3, '1:1': 1.0, '3:4': 3/4, '9:16': 9/16}
    b = bases.get(base_res, 704)
    r = ratios.get(aspect, 16/9)
    if r >= 1.0:
        h, w = b, int(b * r)
    else:
        w, h = b, int(b / r)
    return (w // 32) * 32, (h // 32) * 32

# ── Generate ──────────────────────────────────────────────────────────────────
@torch.inference_mode()
def generate(prompt, img_start, img_end, seed, duration,
             resolution, aspect_ratio, guide_scale, num_steps,
             progress=gr.Progress()):
    try:
        gc.collect(); torch.cuda.empty_cache()
        progress(0, desc='Starting...')

        dur_map   = {'2s (49f)': 49, '3s (73f)': 73, '5s (121f)': 121,
                     '8s (193f)': 193, '10s (241f)': 241}
        num_frames = dur_map.get(duration, 121)
        width, height = get_res(resolution, aspect_ratio)
        seed = int(seed) if (seed is not None and seed >= 0) else random.randint(0, 2**32-1)

        from PIL import Image
        img_s = Image.open(img_start).convert('RGB') if img_start else None
        img_e = Image.open(img_end).convert('RGB')   if img_end   else None

        total_steps = [8]; current_step = [0]; current_pass = [1]

        def cb(step, latent, is_start, override_num_inference_steps=None,
               pass_no=None, **kwargs):
            if is_start:
                if override_num_inference_steps:
                    total_steps[0] = override_num_inference_steps
                if pass_no:
                    current_pass[0] = pass_no
                current_step[0] = 0
                return
            current_step[0] += 1
            frac = current_step[0] / max(total_steps[0], 1)
            if current_pass[0] == 2:
                frac = 0.73 + 0.22 * frac
            else:
                frac = frac * 0.73
            progress(min(frac, 0.95),
                     desc=f'Stage {current_pass[0]}: {current_step[0]}/{total_steps[0]}')

        result = ltx2_model.generate(
            input_prompt=prompt, image_start=img_s, image_end=img_e,
            height=height, width=width, frame_num=num_frames,
            fps=24.0, seed=seed, callback=cb,
            VAE_tile_size=256, guide_scale=float(guide_scale),
            sampling_steps=int(num_steps), guide_phases=2,
            n_prompt='', enhance_prompt=False,
            # Required since a recent Wan2GP update; 1.0 = full t2v strength
            input_video_strength=1.0,
        )

        progress(0.97, desc='Saving...')
        # Unpack result: may be a dict, tuple, list, or raw tensor
        if isinstance(result, dict):
            for _k in ('video', 'frames', 'samples', 'output', 'images'):
                if _k in result and result[_k] is not None:
                    video_tensor = result[_k]
                    break
            else:
                video_tensor = next(iter(result.values()))
        elif isinstance(result, (tuple, list)):
            video_tensor = result[0]
        else:
            video_tensor = result
        if video_tensor is None:
            return None, 'Error: No video produced'

        video_tensor = video_tensor.cpu()
        ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
        out = str(OUTPUT_DIR / f'video_{ts}.mp4')
        video_for_save = video_tensor.unsqueeze(0).float() / 127.5 - 1.0
        save_video(tensor=video_for_save, save_file=out,
                   fps=24.0, normalize=True, value_range=(-1, 1))

        del video_tensor, video_for_save
        gc.collect(); torch.cuda.empty_cache()
        progress(1.0, desc='Done!')
        return out, f'✓ Seed: {seed} | {width}x{height} | {num_frames}f'

    except Exception as e:
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()
        return None, f'Error: {e}'

# ── Gradio UI ─────────────────────────────────────────────────────────────────
CSS = """
.gradio-container { max-width: 1100px !important; margin: auto; }
footer { display: none !important; }
.brand-header { text-align: center; padding: 20px 15px; margin-bottom: 15px; background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%); border-radius: 15px; }
.brand-title { font-size: 1.6em; font-weight: 700; color: #ffffff; margin-bottom: 5px; }
.brand-subtitle { font-size: 0.95em; color: #c0c0ff; margin-bottom: 12px; }
.social-buttons { display: flex; gap: 10px; justify-content: center; flex-wrap: wrap; }
.social-btn { padding: 8px 18px; border-radius: 25px; text-decoration: none; color: white; font-weight: 600; font-size: 0.85em; transition: transform 0.2s; display: inline-block; }
.social-btn:hover { transform: scale(1.05); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.instagram-btn { background: linear-gradient(135deg, #833AB4 0%, #FD1D1D 50%, #F77737 100%); }
.tiktok-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }
.footer a { color: #667eea; text-decoration: none; margin: 0 8px; }
"""

with gr.Blocks(css=CSS, title='LTX-Video Generator') as demo:
    gr.HTML('''<div class="brand-header"><div class="brand-title">🎬 LTX-2.3 22B Distilled 1.1 Q4 — Video Generator</div><div class="brand-subtitle">Created by <strong>TheBuildAI</strong> | Lightning AI L4 GPU</div><div class="social-buttons"><a href="https://youtube.com/@thebuildai" target="_blank" class="social-btn youtube-btn">▶️ Subscribe on YouTube</a><a href="https://instagram.com/thebuildai" target="_blank" class="social-btn instagram-btn">📷 Follow on Instagram</a><a href="https://tiktok.com/@the.build.ai" target="_blank" class="social-btn tiktok-btn">🎵 Follow on TikTok</a></div></div>''')

    with gr.Tabs():
        with gr.Tab('🎬 Generate'):
            prompt = gr.Textbox(label='Prompt', lines=3,
                                placeholder='A cinematic shot of...')
            with gr.Accordion('🖼️ Image-to-Video (Optional)', open=False):
                with gr.Row():
                    img_start = gr.Image(type='filepath', label='Start Frame')
                    img_end   = gr.Image(type='filepath', label='End Frame')
            with gr.Row():
                seed     = gr.Number(-1, label='Seed (-1=random)', precision=0)
                duration = gr.Dropdown(
                    ['2s (49f)', '3s (73f)', '5s (121f)', '8s (193f)', '10s (241f)'],
                    value='5s (121f)', label='Duration')
            with gr.Row():
                resolution   = gr.Dropdown(['1080p','720p','540p','480p'],
                                           value='720p', label='Resolution')
                aspect_ratio = gr.Dropdown(['16:9','4:3','1:1','3:4','9:16'],
                                           value='16:9', label='Aspect Ratio')
            with gr.Row():
                guide_scale = gr.Slider(1.0, 8.0, 3.0, step=0.5,
                                        label='Guidance Scale')
                num_steps   = gr.Slider(2, 8, 8, step=1, label='Steps')
            btn       = gr.Button('🎬 Generate', variant='primary')
            video_out = gr.Video(label='Output')
            status    = gr.Textbox(label='Status', interactive=False)
            btn.click(
                generate,
                [prompt, img_start, img_end, seed, duration,
                 resolution, aspect_ratio, guide_scale, num_steps],
                [video_out, status]
            )

    gr.HTML('''<div class="footer"><p style="font-size: 16px; margin: 5px 0;">🎬 Created by <strong>TheBuildAI</strong></p><p style="font-size: 14px; margin: 5px 0; color: #9ca3af;">Free &amp; Open Source | LTX-2.3 22B Distilled 1.1 Q4_K_M | Lightning AI L4</p><p style="font-size: 13px; margin: 10px 0;"><a href="https://youtube.com/@thebuildai" target="_blank">YouTube</a> | <a href="https://instagram.com/thebuildai" target="_blank">Instagram</a> | <a href="https://tiktok.com/@the.build.ai" target="_blank">TikTok</a> | <a href="https://thebuildai.tech" target="_blank">thebuildai.tech</a></p></div>''')
demo.queue(max_size=3)
demo.launch(share=True, allowed_paths=[str(OUTPUT_DIR)])


---

<div align='center'>

### 🎉 Enjoyed this notebook?

If this was helpful, please **upvote** and subscribe for more free AI tools!

  <a href='https://www.youtube.com/@thebuildai?sub_confirmation=1'>
    <img src='https://img.shields.io/badge/YouTube-SUBSCRIBE-red?style=for-the-badge&logo=youtube&logoColor=white' />
  </a>
  <a href='https://www.instagram.com/thebuildai/'>
    <img src='https://img.shields.io/badge/Instagram-FOLLOW-E4405F?style=for-the-badge&logo=instagram&logoColor=white' />
  </a>
  <a href='https://www.tiktok.com/@the.build.ai'>
    <img src='https://img.shields.io/badge/TikTok-FOLLOW-000000?style=for-the-badge&logo=tiktok&logoColor=white' />
  </a>
  <a href='https://github.com/cafermutluozkan'>
    <img src='https://img.shields.io/badge/GitHub-FOLLOW-181717?style=for-the-badge&logo=github&logoColor=white' />
  </a>

  <p style='color: #888; margin-top: 15px;'>Built by <strong><a href='https://www.thebuildai.tech/'>TheBuildAI</a></strong> 🌍</p>

</div>